# 09 — Option, Try & Either

Real code is full of operations that **might not produce a value**. A map lookup may miss. A string may fail to parse. A network call may fall over. In Java or Python the usual answers are `null`, thrown exceptions, and sentinel values — three different mechanisms, all of them invisible to the type system.

Scala's standard library gives you three types that turn each of those situations into an ordinary value. The compiler then forces you to handle the missing or failed case before you can use the result.

- `Option[A]` — *absence*. Either there's an `A`, or there isn't.
- `Try[A]` — *failure as exception*. Either you got an `A`, or some `Throwable` was raised.
- `Either[E, A]` — *failure as data*. Either you got an `A` on the right, or an `E` on the left describing what went wrong.

All three are sealed two-case enumerations under the hood, so everything you learned about pattern matching in notebook 08 applies directly. The new skill on top is **combinators** — `map`, `flatMap`, `getOrElse`, `fold`, `recover` — that let you transform the value-inside without ever writing an `if x != null` check.

## The same shape three times

Before we dive into each one, notice they share a structure. Each is a *container* of zero-or-one element, and each has two cases:

```
  Option[A]      ::=  Some(a)         |  None
  Try[A]         ::=  Success(a)      |  Failure(throwable)
  Either[E, A]   ::=  Right(a)        |  Left(e)
```

On the success side they each hold one value. On the failure side, `Option` holds nothing, `Try` holds a `Throwable`, and `Either` holds whatever error type `E` you chose. That difference — *how much you know about why it failed* — is the whole basis for choosing between them.

## Option — the cure for `null`

A reference that might be `null` is a value the type system lies about. The compiler tells you it's a `String`, but at runtime it might not be. Every reader of that code has to remember to null-check, and forgetting is a bug.

`Option[A]` lifts the absence into the type. A value of type `Option[String]` is honest — it announces at compile time that you must consider both possibilities.

In [ ]:
val present: Option[Int] = Some(42)
val absent:  Option[Int] = None

// You cannot use these as a plain Int — the type system blocks it.
// present + 1   // does not compile

### Three ways to construct an Option

- `Some(x)` — wrap a value you know is present.
- `None` — the absent case.
- `Option(x)` — the *smart constructor*: returns `Some(x)` if `x` is non-null, `None` if `x` is null. This is the bridge from any Java API that may hand you a null.

In [ ]:
val javaResult: String = null   // pretend this came from a Java library

val safe: Option[String] = Option(javaResult)
// safe: Option[String] = None

val also: Option[String] = Option("hello")
// also: Option[String] = Some("hello")

Many standard-library operations already hand back an `Option`. The biggest one you'll meet daily is `Map.get`:

In [ ]:
val ages = Map("alice" -> 30, "bob" -> 25)

ages.get("alice")    // Some(30)
ages.get("carol")    // None

### Taking an Option apart with `match`

The most direct way to handle both cases is pattern matching. Same shape as notebook 08.

In [ ]:
def greet(name: Option[String]): String = name match
  case Some(n) => s"hello, $n"
  case None    => "hello, stranger"

greet(Some("alice"))   // "hello, alice"
greet(None)            // "hello, stranger"

Because `Option` is a sealed two-case enum, the compiler checks exhaustiveness. Omit the `None` branch and you'll get a warning that the match isn't exhaustive — exactly the property that makes `null` so dangerous, fixed at the type level.

### `getOrElse` and `fold` — the safe extractors

Pattern matching is fine, but for one-liners there are two combinators that read better. `getOrElse` provides a default for the absent case. `fold` lets you transform either side into a single result type — useful when the success and absent cases need different shapes of computation.

In [ ]:
val ages = Map("alice" -> 30, "bob" -> 25)

ages.get("alice").getOrElse(0)    // 30
ages.get("carol").getOrElse(0)    // 0

// fold(ifEmpty)(ifPresent) — note the two argument lists
ages.get("alice").fold("unknown")(age => s"age is $age")
// res: "age is 30"
ages.get("carol").fold("unknown")(age => s"age is $age")
// res: "unknown"

Avoid calling `.get` on an `Option` — it throws `NoSuchElementException` on `None` and defeats the whole point of having an `Option` in the first place. If you find yourself reaching for it, you almost always wanted `getOrElse`, `fold`, or pattern matching instead.

### Staying inside Option — `map`, `flatMap`, `filter`

Often you don't want to unwrap an `Option` yet — you want to *transform the inside* and stay in the container until the end of the pipeline. Three operations let you do that.

- `map(f)` — if `Some(a)`, return `Some(f(a))`. If `None`, stay `None`.
- `flatMap(f)` — like `map`, but `f` itself returns an `Option`. Avoids `Some(Some(...))`.
- `filter(p)` — if `Some(a)` and `p(a)` is true, keep it; otherwise become `None`.

In [ ]:
val ages = Map("alice" -> 30, "bob" -> 25)

ages.get("alice").map(_ + 1)              // Some(31)
ages.get("carol").map(_ + 1)              // None — map skips on None

ages.get("alice").filter(_ >= 30)         // Some(30)
ages.get("bob").filter(_ >= 30)           // None — 25 didn't pass

// flatMap when the next step also returns Option
val nicknames = Map("alice" -> "al", "carol" -> "caz")
ages.get("alice").flatMap(_ => nicknames.get("alice"))   // Some("al")
ages.get("bob").flatMap(_ => nicknames.get("bob"))       // None

### Chaining lookups with `for`

The reason `flatMap` matters so much is that real code chains several may-fail operations. Each step depends on the previous one. Written with explicit `flatMap` it gets nested and hard to read; the `for` comprehension makes the same code linear.

Both versions below do the same thing: look up a user id, then their profile, then their email — bail out on the first `None`.

In [ ]:
case class Profile(name: String, email: Option[String])

val idByName    = Map("alice" -> 1, "bob" -> 2)
val profileById = Map(1 -> Profile("Alice", Some("a@x.com")),
                      2 -> Profile("Bob",   None))

// nested flatMap — readable but starts to drift right
def emailV1(name: String): Option[String] =
  idByName.get(name).flatMap { id =>
    profileById.get(id).flatMap { p =>
      p.email
    }
  }

// same logic, linear shape
def emailV2(name: String): Option[String] =
  for
    id      <- idByName.get(name)
    profile <- profileById.get(id)
    email   <- profile.email
  yield email

emailV2("alice")    // Some("a@x.com")
emailV2("bob")      // None — bob has no email
emailV2("carol")    // None — carol isn't in idByName

The `for` body desugars to exactly the `flatMap`/`map` chain above — `<-` becomes `flatMap` (except the last one, which is the `yield`'s `map`). Once you internalise this, you'll use the same shape for `Try`, `Either`, `Future`, and any other type that has compatible `flatMap` and `map`. **The for comprehension is the universal sequencing tool.**

## Try — turning exceptions into values

Some operations can fail by throwing — parsing a string, dividing by zero, calling out to a Java API. `Try[A]` runs a block of code, catches any non-fatal `Throwable`, and gives you back either `Success(a)` or `Failure(throwable)`.

Where `Option` answers "is there a value", `Try` answers "did this throw, and if so, with what". You import it from `scala.util.Try`.

In [ ]:
import scala.util.{Try, Success, Failure}

Try("42".toInt)        // Success(42)
Try("oops".toInt)      // Failure(java.lang.NumberFormatException: ...)
Try(10 / 0)            // Failure(java.lang.ArithmeticException: / by zero)

The block inside `Try(...)` is evaluated lazily *by name* — that is, only when `Try` calls it, inside its own try/catch. That's why the divide-by-zero above doesn't blow up the line; it's captured.

### Pattern matching a Try

Two cases, same shape as Option.

In [ ]:
def parse(s: String): String = Try(s.toInt) match
  case Success(n) => s"parsed: $n"
  case Failure(e) => s"failed: ${e.getMessage}"

parse("42")     // "parsed: 42"
parse("oops")   // "failed: For input string: \"oops\""

### Combinators — map, flatMap, recover

`Try` has the same `map`/`flatMap` you saw on `Option`. The new combinator is `recover`, which converts a `Failure` into a `Success` by handling specific exception types — like a `catch` block, but as a value-level operation.

In [ ]:
import scala.util.Try

val doubled: Try[Int] = Try("42".toInt).map(_ * 2)
// Success(84)

val failed: Try[Int] = Try("oops".toInt).map(_ * 2)
// Failure(...) — map is skipped

// recover handles specific exceptions with a partial function
val safe: Try[Int] =
  Try("oops".toInt).recover { case _: NumberFormatException => -1 }
// Success(-1)

// recoverWith is to recover what flatMap is to map — recovery itself returns a Try
val retried: Try[Int] =
  Try("oops".toInt).recoverWith { case _: NumberFormatException => Try("0".toInt) }
// Success(0)

### Non-fatal only

`Try` only catches *non-fatal* throwables. Things like `OutOfMemoryError`, `StackOverflowError`, and `InterruptedException` are considered fatal — they indicate the JVM or thread is in trouble and should propagate up to the runtime. This is the right default; you should not be writing application code that swallows an OOM.

Treat `Try` as a value-shaped replacement for try/catch around *expected* failure modes: parsing, I/O, third-party calls. For control-flow that branches on a known reason, prefer `Either` — coming up next.

### Bridging to Option

If you don't care *why* something failed, only that it did, collapse a `Try` into an `Option` with `.toOption`. The `Throwable` is dropped on the floor and `Failure(_)` becomes `None`.

In [ ]:
import scala.util.Try

Try("42".toInt).toOption     // Some(42)
Try("oops".toInt).toOption   // None

## Either — failure as data

`Try` is convenient when failure means "some exception got thrown". But often failure is *expected* and you want to give it a *name* — "user not found", "invalid email", "insufficient funds". Those aren't exceptions in the runtime sense; they're business outcomes that the type system should make visible.

`Either[E, A]` has two cases. By convention, `Right` carries success (the value of type `A`) and `Left` carries failure (the value of type `E`, often your own ADT). The mnemonic is *"right is right"*.

In [ ]:
val ok:  Either[String, Int] = Right(42)
val bad: Either[String, Int] = Left("missing value")

ok match
  case Right(n) => s"got $n"
  case Left(e)  => s"err: $e"
// "got 42"

### Naming your errors

The real win comes when `E` is an enum — you enumerate the *known* failure modes and the compiler then enforces exhaustiveness wherever you handle them. This is one of the most useful patterns in production Scala.

In [ ]:
enum SignupError:
  case EmptyName
  case InvalidEmail(value: String)
  case AlreadyRegistered(email: String)

case class User(name: String, email: String)

def signup(name: String, email: String): Either[SignupError, User] =
  if name.isEmpty then Left(SignupError.EmptyName)
  else if !email.contains("@") then Left(SignupError.InvalidEmail(email))
  else Right(User(name, email))

signup("alice", "a@x.com")    // Right(User(alice, a@x.com))
signup("", "a@x.com")         // Left(EmptyName)
signup("bob", "not-an-email") // Left(InvalidEmail(not-an-email))

Now any code that consumes the result has to handle every named error. Add a new `case` to `SignupError` and the compiler will point out every `match` that's no longer exhaustive — the same change-impact-analysis property you saw with pattern matching in notebook 08.

### Right-biased combinators

Modern Scala (2.12+) treats `Either` as *right-biased*: `map`, `flatMap`, and `for` operate on the `Right` side. A `Left` short-circuits the whole chain, just like `None` does for `Option`.

In [ ]:
val a: Either[String, Int] = Right(10)
val b: Either[String, Int] = Left("nope")

a.map(_ + 1)      // Right(11)
b.map(_ + 1)      // Left("nope") — map skipped

// fold(ifLeft)(ifRight) — collapse to a single value
a.fold(e => s"err: $e", n => s"ok: $n")    // "ok: 10"
b.fold(e => s"err: $e", n => s"ok: $n")    // "err: nope"

### Sequencing with `for`

Multiple `Either`-returning steps chain the same way as `Option`. The first `Left` short-circuits and is returned as the result of the whole comprehension. Crucially, **all steps must use the same `E` type** for this to compile cleanly.

In [ ]:
def parseInt(s: String): Either[String, Int] =
  s.toIntOption.toRight(s"not a number: $s")

def positive(n: Int): Either[String, Int] =
  if n > 0 then Right(n) else Left(s"not positive: $n")

def total(a: String, b: String): Either[String, Int] =
  for
    x <- parseInt(a)
    y <- parseInt(b)
    _ <- positive(x)
    _ <- positive(y)
  yield x + y

total("3", "4")        // Right(7)
total("3", "oops")     // Left("not a number: oops")
total("-1", "4")       // Left("not positive: -1")

Two helpers worth knowing in that example:

- `s.toIntOption` is the `Option`-returning version of `toInt` — no exceptions.
- `Option#toRight(left)` converts an `Option` into an `Either` by supplying the `Left` for the `None` case. Its sibling `toLeft` does the inverse. Together they're the bridge between absence-shaped failure (`Option`) and named failure (`Either`).

## Choosing between them

There's no rule the compiler enforces, but there's a clear convention:

```
  Option[A]      use when absence is the only "failure" and no reason is needed
  Try[A]         use when wrapping code that throws and you just want the value-or-exception
  Either[E, A]   use when failures are part of your domain — give them names
```

Concretely: a `Map.get` is `Option` because "missing key" needs no further explanation. A call into a Java parser is `Try` because anything could be thrown and you don't care to enumerate it. A signup flow with five named rejection reasons is `Either[SignupError, User]` because the *kind* of rejection matters to the caller.

## Conversions cheat sheet

You'll routinely cross between the three. The standard conversions are:

```
  Option   ->  Either   :  opt.toRight(leftValue)        opt.toLeft(rightValue)
  Try      ->  Option   :  tryVal.toOption
  Try      ->  Either   :  tryVal.toEither               // Right(a) or Left(throwable)
  Either   ->  Option   :  eitherVal.toOption            // drops the Left
```

The direction that *loses* information is always available; the direction that *adds* it (e.g. `Option -> Either`) requires you to supply the missing piece, which is exactly what `toRight` does.

In [ ]:
import scala.util.Try

Some(42).toRight("missing")          // Right(42)
None.toRight("missing")              // Left("missing")

Try("42".toInt).toEither             // Right(42)
Try("oops".toInt).toEither           // Left(java.lang.NumberFormatException: ...)

Right(42).toOption                   // Some(42)
Left("err").toOption                 // None

## Putting it together

A small pipeline that uses all three: read a config value (may be missing → `Option`), parse it as a number (may throw → `Try`), validate it against a business rule (may reject → `Either`). At the end we collapse to a final `Either` for the caller.

In [ ]:
import scala.util.Try

enum ConfigError:
  case Missing(key: String)
  case NotANumber(key: String, value: String)
  case OutOfRange(key: String, value: Int)

val config = Map("port" -> "8080", "retries" -> "oops")

def readInt(key: String): Either[ConfigError, Int] =
  for
    raw <- config.get(key).toRight(ConfigError.Missing(key))
    n   <- Try(raw.toInt).toOption.toRight(ConfigError.NotANumber(key, raw))
  yield n

def readPort: Either[ConfigError, Int] =
  for
    n <- readInt("port")
    _ <- Either.cond(n > 0 && n < 65536, (), ConfigError.OutOfRange("port", n))
  yield n

readPort                       // Right(8080)
readInt("retries")             // Left(NotANumber(retries, oops))
readInt("timeout")             // Left(Missing(timeout))

Read that pipeline top to bottom and notice what's *not* there: no `null` checks, no `try/catch`, no thrown exceptions. Every may-fail step is a value in the type, every transition is a `<-`, and every failure is an instance of a named enum that the consumer must handle exhaustively. This is what idiomatic Scala error handling feels like once the three types are in your hands.

`Either.cond(test, right, left)` is a tiny helper that builds a `Right` or `Left` from a boolean — saves writing `if test then Right(...) else Left(...)`.

## What's next

You now have three types for *one* missing value. Notebook 10 zooms out to **generics and variance** — what the `[A]` in `Option[A]` actually means, how `Option[Cat]` relates to `Option[Animal]`, and how to write your own generic containers and functions. After that you'll be equipped to reason about every parameterised type in the standard library, not just the three you met here.